# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/719W5PbSJbee/0KLMYTBbRIimRJ1RK1nF1NX2WruxWqnmmH2TQDJEEWJN4aAOuiioqYf+Dn9dOEw+Gdno1Y+8Fhe2P8NPVP5g/4L/j7zskEEiCrVJpuW9FdBPKeJ0+ePHlu8H3/4Kv1Ko/msRdt8/UyypMftrH353/ysvxyEXsv15NoE116f/7TQfM+/w5+m0xvflx743Sbx94smsTeJFre/JhGntf8leflabTKJmmyyZP1ygu+O02yTZyGmomak/V2E3vTOPPGi2g1yQ6Ynq23WTNP8hTJ0SpBc5m3XOfeJkrlN0jyaCHjniUr74GkTRY3P3rxypunUYYhbFgkzkJpL40n0TTFnNHcIo/TFRt84KHGPE69d+v10psAKMlqq+PCsDhsDAvp20W8Tfm4jFJ0KQ0u1vO1Fo1nszjPMOLVmqPVJjPNW24zGSTqztarqResBQqreLGIG950O3mbrOY6wq9ePfLO4jRPJtHC67SftC86T7ttb5Pe/DH3bn7vbbbjRYKxBg9qEG3lF7kX514rS/OwdeBjfZPlZp3m3huMqYFB56cNb501AISGl51u82SB3+14k64ncYb08+gsPpilgME0yqPJIsoyTMQ0UiQ1AOp4MT2wra+2y82lB1CvNlr51YuXttKLJUDd0J9P0+jcPH6eEPb2BfA+OPiu4X3Z8D5/deL1ZdYNj9NueEftg5PXSHv0pN1uHxwc/MLr/4R/XopVWXD5D/6+mNCB/PVex5rTO/Dwz6z2qNPDdkgxAP8X3c+6nzzu+J7z7xfA1XgLECR5sV8CizMPiJNhpbVu2Vr7+Sfd549qrXHhbUNScbPOEq7uiBthpBuh580W6yhHK+3W8bFWbOPtFNsYYMPTOMqk9mo8imUsIzOAnpesWLEr2YqxI4OxxdAUc31nWEXSeuv50XayXfnSwNl6sV3GI4Pd7rg6T6RAtt1s0mQZp6PTGBOJOBV0NF6vFyj2bYotIR3EW6Dm6XbZwNBPW62Wzj3aZmg8unAbfvTYq/77BahXcxovbn7f8EC1hHZ4cZYT5hsQgiDTJcCencej6Cxa5ZWBtt3cTQ28nceSO92mcTzaoOnqcLqtR7eMZLsqSY2MJlllIIU3P0p7pDNlKx3bTQxCBaKAKhUQSR7pzCheYSGT1W6uDnDGrLJVXYJZMjkFvRixgWKJfYNci2QSj7I4TWb7s0hCs9jJOziYxjPvNL7opvNxcBrqZjkl9rVQKNkEYWuhD/4vfIV8GufbdOXl280iDoB/wekg6SXYHt0h0PU4BElMvQQQ8gJs+G7DexSGP32nV4ijjHqEwyEbTePNNgHSx/NlvMqzwD6YqbAMZjMYKvAwMhTg2Gw5LWYzz01Wax7ngX++TqeZ30Dt0EPmYFgW5j8uC2ATnA+kpD8MLcwqxZKZl1crysBQ+co/93seNrmf+Wal2ViWR2mO1pAeu+nxin1c7zSFDs51wCD942icLJL80g+9BDQc5+fX61W827+MYeBv/CEGkq63q2lQdOQ2w2EchbvDB1xb0WaDMQXLCl4wx+AVHkklK0sXmIXBebZcXHjBV9HEe74BKnkn6BCHNfqbRRnOk+a5shRa5sUqjxfeQ+/VJyiwRr9RUhb4ZL2IxmHLN9huTiv9WSTjlp6N2PA5Fnl5YIBm31vZJfpbBiEw1PM/jdLzZOWDPZmWJZYRdt0qNkWidHn8SEtU+2hhy05HGNMk4OxGZoB+WMLfQInZvh3I7Y0oJO5qR0vco6k72lBwahtplGSx93oLpmkZf5am6zTwn/OIMKtJnqfKqIAUgtCRSQswKbsoO6uIs6a6bFgv0oWvvvn0s5efnYy+evlvZUtgSOM10J4Qak7Wy+V2BTx8aCo1F6TszbOjppbj3lmi99srSHaT4G7UsdhfxtNku7y9ruZrZZyaput7DI41roupff785NvPXldnd8c89KFRGZ4+1QZRPF+bDWcXJo2Ds2Qar8ElrnGCxX3f9oIzb77F+yzFyxk4k/F2EaF83/fLjfltZX0NZ++Bm5x4p+t0TU4Lx2CFbzfIMTldJ1kCCoqDD315Zt+UW9OU699CGywa22J93Sglypqt7WyuIgucCQ6jWW30xMmGd6VwuA7/8rv/gonPFtvstM/jtqRsMqiy2ZYd2biA5QYc92gN3mc2SuPNuu9grpBf7cPCPDTABqj6CvXGXiJc+8eTZMS9h0213GQyyAb2GNjGaDECZV5u8r6zcDyYSOHv1TjI61T5T/wH5ugsETY0vsj7n0eLzIWGkob9Z2yKM8o843zYs2aGKDlnKy8SVWJml9Kg11cE2vvWskpS7rWsPGfdLoLqpty/cFNAZoI9wYs02sVG3+ASPMovN0jEyJ74ZQeABO5QI2LPHpypo8D/u/X9mda20s3gyvBAPfuIp/MWH0iwhE9hgjyRXVlpAfzizWUjmOq8X986m4ILCzLpJ1POKxxeK/dmGLTM4p0iUb5OJ6dugkscsJrkdifbaeQLN8bCLb62MHNcIRJwDwse7jHghIKbbXGeSt2+psmBrzgiAyyIqhLiCnUuqbdz3ipW+3/5j7//P//rP3jeq0gEEV+8+o3XY7s3f0DDh0L9Dz0e30nGQ3UDrMZVaCXSgCXuCxmlBz9OkixseX4NkP7zSZ6cxe94Y9GGP7tA2a3sn1952ALJjOIG9ERs9qaHsZv/7SPWajnobebbt8fSgbs9PznlfIk4uLIUU7AHBkRPdoNC6oTLY+pdAZ7X3KzmEod2Lb0F8zkdSfFiP5ptiB/DYf6kTbYf5e6x87zZpnPcDwpUIB6FtzT3vl3YoExgvM5id1N+CLE9OPjys5MX3z7/9sU3X58IS4HLNtHtNN7yB2/6quzC6VJ+lktJ1L+4k8uPqRifgn84+fabV7zO8Bbh+1hEHN4Lir2ISNuVCvKArFjmKfEQ3NwSnKI3XeN6DkCtEojmUu8N0GrrJQuPkjAPKLXC9MFn4E+yyCQ1O1iiUIyuiMzYjIm32OJ/nh38k3mXlPZR3Ih+8H/Op8y7+ccIXXgT7weUpvDthy3+Q20MAeO5+ReOKaJILcLv9uKA1/SMeRngiUs7XpZLNvTHnOLChDKu6AwCDHI27zyWxDw3SJUdl6c3f+DfmH/WG1yhMUDyO8IHZZQIEaMpyDmY8m1yilbOKCNcYuI5/kfvgJKXRyjusWt2TBkcrmXArzP5u1oTQEhPeM4ANhAaoA0IS5AyTmI2tc0PcsKQT7H+zXBrBxFaAxgrdHO2TiCokB8sHJ7wk0wSThZ9YJpJivMXYq2ZoPbkUKBziJX6YWv+cnEO3hwCMCvJnR5i6fkHyHCILjD3ZYKl0QYhd/EgXyGMRMziKd56vBEyL15NIIWiIPgNRjON30TeGe4WxOODN1sc497y5o+Y7RLkA72TtMr0UZWQlpXdxNucfy4AVv2DUSX8P42Bpa1sg7MkgHTh8+cvfo0z/a9BYOKLwRii3dSu657lLhHvAO2tiOxv3o/uu5g8+SAEObAIcsktA+QCzLh/chdDBWMLDHJXqsDKCsD0ugA54TLIzQFliFAatyBDDlJ/8O+/P7/5/c1/uvnPNz/e/OHmjzf/dPPPN//15r/f/I+bf7n5083/vvnHw7/87h+aQ5IQ/J+DgJ/HKRpvgTleQF2Ag+53/8DcQ992CIGBSPdTyL1SCNREfsi7Q0Ymq0gpbyJOIo6ywwVFqTc/9o1I9RnRC6I+zHGb9p3n7NDeN3Ke7CST14UAaCM7INGpCkAw2Wffr4Z+ZRCkar57eoMb8Ps+K0oLVZFKBFEnupEc0yoKQx62IwuKrJBoVyYjgx3IqhSlQspnxhXBUjITuY6U3rnMi/iF7yASVcHXkkNndlnnLUpId5AEnVtW3nTxlsVrfagocTVhy5gnpQvgCQi/Rutv/u5ZDyf78MG/wrTr7dnDfppMwG7jgO7rXN8OITcMpMlBeyhCGWleGTE/LFvAdPbLm0gcze0Xp2xcu1FyvlUE9z2/9QZclJmz3FxUepblGLncHkz/A87HrwLvZxBlnkHFNRXZuRAlnDiZqDBOPvvNi5ejT775+vMXz7/+5DNVCJi5jd7EOfeKzLE6oUEuI8yFJ5XVvFAZ7IWsICsUG1/YX+Fjc8EhFHDYiaHdpqtocQkxstmZWbxNFv3a6Mo9+qnV9YHiViVDoKebU4icMUkQwCyZo12SQCGLPK0gQld2VBtrWlhgXLxzpaDPATiqWQIlQMlX4gjjpu5RNZng2oFbWhobyYahgAtRzGH5Qu+ZbTqlYAuMMwhIfvNj7AWUT8gZpIMUuX4Kph3EVU8oc771SNnBH0J5Iyq/eZROyUJHeElXyc0f0rjsZsaDBKQINBhiETyMeWgVfURjAB9KATD2clyCVqZk5C29MvACJRIpyWDYcLdw0lBEjKGki9Moj138NvsWFWsbNpX7D0sOkDlodobDOlHiZgB7KWCKzyBqxvOviH6kgqRoMSkrNzr2OXcFN7kW3dnnxY5NSQZiabXovD0cak+/8jrdPSTQzN9udVQJnxWwKEojwZZICppYmfiedg72k8W3uH1PL6pQNbWd8Z2LFkGmkQxLDQdqDp3r3Xosxc5Vrl5cZVHZkdFXJfPOpIDo5GC2y0BaCiHtxrXPvpA2SgdCm1i1lK7IhshM5Y6ecnJQSQXW9P4Wq3n0+M4220WD2KZpHmdVoBPEHKIzeKEkskPRvpAJrrodzq/62mftpNS27cr4zlaX/SxsvLuj/y709x4DV/4KYoW3OEA6YDKm8ZhChgwYJgiGJBxPmgSMF0REGhYMafi7//7mC7mkbNWeEud6SjhrSb0Mp4xS/MGbmRISzNMtjSvhQDHCzxalHKWadG1Q9RdKszKHaGVYP5fCkDgRfBHF9qBSwqwS9kcF5cXth8x6rFyBmgMksxlUBFZWchKDJQOB/SrKwT2l7s6IqvsC0HdWEyx9vziYooGBXZWLILK9iUKgR7eKBsYsI66o38aeXGnygaxqj38f1ZRuaHM8kNUmXUGvXGWSk+7jXWoyTuPobSX1zdgZ83jPmEWkl5BXqYElUFnAG3B6b6BrSnmA1zR9GNU2HylJRwPF3M3jOJT98mY86JmsIQUKb6IdaswByO45lhq2ISQdiSbS7Wh31gCKxcih3SwzX0vz2JQzrjycvavx4HB1OLz2LWfpNn8fsbKSj5lfOft65mzb11G4b8xmd5DfFdnIHatZYf0++eblN19/DYnub5+/fPGpcDIkXP7Xf/5vvH18Kkcwnz4HpuDnWy45NRYUhZELDZn6iWUz+HLzu5I34fsXOjQ8nZQGTL5llxZgbOJsVLJ1tfPD8mmDzQAkC1RI1a0bg8a8JDhpgs+atinws0ZQfFkqZAsdGpIkCzk263CljXXabe8jlDpkqUNojr1f1nVghTDSx1bzvT//T0v2Ng4OKdvoplj+XIZo101eQMuGQ+cAUkBYSI0oAgB7WoELVegXYWluoNdIy55SB4rOCPs2/0DYoT/m3smEeM8F01kN5WILzkqXq+Rfn5s6QjepwLogv7Y4VAlsJEqNXmn60rD8n4qASnxo8OSS9ZJrZDI31iFBfNEiYwyzMOGKZ+CRIcziXsx5m41V1rBdFdwvdsyMJl6pbqUkFokE+c1tyvKQYZiiJeO4w9dsTs1UCcV3AG0NAEauXpIPLdwHX5xkeSBvIbBiIFwKEOlJOOg9KdkBmakFBi4J1hpCKg6Ohub+3DCLrsnHw1rCx9XjgouuLb7ntDBls5plRYU9oHGFoAhP5s1psd3EqoLvZqvpASw05zp8b7fVmxgJuE1hk/Y82R0WMzAsvQErH+mYyrAueZPhewYAAdNZTDLbr97s6oCx5f7/gmezTqbCNfLEOif2dEvOyQ7JQaE1DAcaakdDrpe1SfbsWKqmOcg4t7hs22polzVwT4nFxQx4qbEthiSI52CipOu7YXNuQKPENKcZjAGOScH0pmiv3XraZWYNOMiGvMMycoYZsxZl3MIiAYHkMciogEkT8HchCA3RXzi9XO/JoEwiNS/3tUNYBdDV/ezwSx+MdpVz9SdKOWheJ8a1apQs5BkIAxKxhYxMrO8MZRZrt4b3ulfYbNaMt3TwYVUQI4Ibe1Mtrq+duHk0tDfC1639BosHrhWT3Ot2m16JVcOOYIjsmCsxM8eW3+StriJMCYeusK4qd1Mjm99Gi23FxEYkt+uFSM1ziCGF1vOUAek/E2tsq5m7RZ5n4IHLfnQRLM2dx8DmARC1/aS4WsDCG3aLKxWKm24DnE5GlFOYbQO4heFmaKwdV3HNpu4+cgmtJipUu1iSpLcz7FSqv5AMbvl1q7QVreypvRX6ZtrP3Gz2X3K/Sbk1ybnsabTc+qAT0nDPjFMvkmWS3iOlfWjFk+F1uSLvhDyxubKHd2CbiiXBCjSYYvsgEF63HFPW0K031nrgxswOeedO+0FZk/oXM4ptbWX2DQmLwXJcCjM6wBwpCrmxxZXaha1SQKfjJjXMiCGuKZJ1EUgH39mXO9YB1ewqvKtiGnLKCQHX3lVxTYqFVUNOoW5Fr9WODBkc+Kwok3mXuNuCpAnSiH2+BQ2zP0hXYPFGdZVoE6B6EXonmkLYPs2NnfjlRLQPgw4X/nWLhsJg7VvtR/ZNQW1Ilnb83vXjtVsX7pmIrvaBV+7vouSAECUOOg25RKJ0WDsrl6Klboh0XgEzvcD1G5R0SNpRpOzKDIttHIFGvG5VjarlXPz4sUUyva0vregPWbvX1olwsIEOSAnZA1OFYqvurqyQ4LIoM4goM82Jf4QPHg9uLacDMmdeAfQrQLRHRY5PSdIYv1wgSnm4ioPE+6XAUN6sOQqQMaDqJ6zJD9mmOQDU8rxdnt9y7rKAI60c+GtBxPyZ8g0bC7KNrLNxOkg3HGju4uo8JT4KEXeuIlLAZP0EUbIcXPUjsGet4+1tRRmWnetQHV125NLOGN8j7r2NFf9wUfdqAosVw25yMaXkm6Ge5QKXNwQKx0E3ongV7NNk3SEwP2rvSMxL1dj9hOaCZBwAxD2PxLZCBg0J1yMjUzd062/6pjV9/+uE8K0n4V+7LO+Xwu9px+CusbTAcZbI1VdLyj2XF+EYOKSGpTAiIjPy5z81aE8ENZJ5QgN//pMoeHpUwBMZ6fhWNiRSzwLd50lJDAlgM7KQhM4BwLzhrUhQTfZgnmDXlC+USpbzh+USmSZpLiQsHe6QoJ4L/hncQu/GSqEOba2U34U0O83ta6NSyB5tZjOsLnTR92NKUdsWy0p8sQ2yNdEhtGv4srpowSQNgiF6e8xbm/VGDAyY9UX1LJsne1QtdiE+TNXyjoeecHtG4q8TKwuAw6JtWaEOKC/QF7mlELV1LacONNY0zLiCLCL26sRPy2MWA1MWDe02XF2DHOzHBVdUbhoAWyYoeCPIoiqGELfLt6EVvJ9XoXSe7SEXibHgMPWtxExMuazYuNKBUBRn/wpvQkZOBhUOOjJ1HeGulumLkkc23O95Vihe8LfQuOAvVSQLXpUntOPzxWsNb2zvunIF2eOQGSwhd4MX3jQu7z3FJmSpNDfQWFm2mga0efC6teMgF5YWEwVIvhA4HJcIl6N+Tj+7LwRbjBjki+Ke4WAPYcPFS0q8VJrSbdi2m15X5KXi2/a3bDQZlpIIfVP1xd/S46ztNG61eKtNa4E9taERDXdcg6iEHlahzoSkRtZnwE5cBWRS09lxlBM50aTtmv/ShNxEG5g6AbsWAARNvFYRDWqiVG97zpQFCjRlge1HNM6ChMegUkBH9ZhYDqjGUWiT91HWjGORKnJ7aSXskPgSVp7L8RSaKTBr6DxwQfrABbdwjZzTJAxrbEJSnEnsw9mdijYyYesV97pVOI9ii+6mdsMa291QKJW7N4OWLYZwnyCpUU8dru4PMoB2AGDBfwmHugNXLFPsOxWbCM3jq/zS3ppMJWV3/MW72TJI+UJUlmAfRWeJ36oawBdpmM2Tm2XhgWhdS7J00oVcSFhbbBczjQpDOzAjGFaYSOVgVTRzTJzPQVWVv62CwszS8MJwh66yvyZbSNJP9yakWExVEuoaGUDFtyTdwn1OnLjPojQRcRH9qyFjNRPO00vnookFK3yeYZe8jWlTXWuqRK5KXamPAz8fFT2NxpejVbSEWbNNccQWF5N4k8OYmz/IqfG9dHuuwXF2cHc9U2zP+GW+5fB1/ROWo+ghMIL/mYEIBbiAkdq/zXhbGI/XF0Y8bhx/STOP9AoEqSrdS1Ms7kLkwx8xn7WbvPrYBNVpLLHpLYxbq/g88F+CGwrYBraiIVWFN3qLfwKpFLbYfxCw6yaHqA/s4BFq2ilg2v0ZHeAXi37nsWnP9iq/sHHj3T9w/NxbX4htbrT69WKbBk/tEbOsDvT1F79+7oyVZ5T5LwQvnSxb8CiJFpvTyAxYsUuYk3u2sm/yaOC2qd8y7aD7+DG8dN0/xYxaMsARDQzotx5L8+5mTJYNXb1Uu5JFBMKoB/4r2VmG0steG9HafjSCk+4C/b92aCGzcZlZQuEafDQ5DfeShpVMbOKcaqekMOusRdcsiLqhuMqCCTmpySnOctflBSg9Ilm3nfgPT8Tr8+FLnCxRevmQeyB7eELBjng0RIuHX8TrdJ5E3gsGpZi08nzm3+WG4z/cZunDDJe1+OFMWrNb6iHtn8+2Dz/Fz2+3J3TQbjqN1sc5/sBxPge5WHi/Xi+mP98YQcibRYuut9Ni1tqIUSAfxuYhtSkpxw75reOK3ijf1f+8YZfDPoyrzU/gNGhsdgvUoRBCsUaimRTUqWrLGlQzq+ashj0uu9gxLNFAKfTxGPu7/ElZkWar/eLkKCBhjw7/swuoewk8v4BKlQGJF5XeJh/cW+b09pX4GRkkLXrM6j1m8U/p5PbWrRbfbcr6f0PLwSAHG9CoADzI64YJROBo319Bo91EbJTJVm32bYQM3iJoLcqLwWlE9byIAyRaBWRNkBWq04A6JlPxDxuom3/OS424XDm54l84/Ah20q5Z27zkwIwlRRVUvOcQveZWzNrgIy84Owzu2/0tFPp16XxgbmSiYMRdz8g4HRuzmthq/+qpEwavku9rbtCry2/L+98gKE3+G61nPTXgpmIY9d4aeUML5IZWGSIcLXLYzW6z6g5yr2GpmdeeoZWwWkiFhZiCyIQbeqdUpYeYmFWr8z6W7coPpRVrDqHN70JU7KgF5UcF8o/EKC0IGMPmCZmH4PhJw/u4y6fHeDqWp0d4etwNw/2BFhJFusBhooqtIPLVLHe7m4k1hZAhvefZMfFky7PwLpssmQKpJEsWc90vHIyzDYbFq3O71X0CxqvoZ2/xxbmRnCa8J3dd2cwyo7yU7X2kAjFJadbdH1zDwHPeBb4TPcGT496tU9q9FRaLbK9w7AzX7/OGCANkcEe1wTU4ttotEOqdVKK+PKreik/NLFM7xdRe83WSUs1M06TuTPTSwPVLFOTptxv1h5XQ1cO6YkOGbeezOJWBF/3vgonCyuA7cl8wt3i4T0vCBoOCUTsnJ3kaWuRAT7dgLAmlA+AGempqE5d02Vic4hUN2URg5f6lvqA64/xcsWOnxCVz0ZYB6x3aQWudVFKVHYJSkOfdna+ILRv5KYRyTyBM+Rj7+bi9Z8O+d7P6PAcNi6Eb0u66O3dcZbcdF7ttF2w/zz675x7b3V+LUyNW27uTKkUvGoLpFQRE4C/5ZdIt+P1h6PizoOJ+NATGSOsiepEna6gYw7qmMFS8MEIoAuWCwrh2KFrcyl2I0rOAf+B989FH3pFpabLGGqcjBruLKWc9i7C3ZRIYPqBgulAbfWgK6xd0vcWTN5MhbXB5zCaiUnzSFRlzh4Q7Vq0nBymqZuJHp3VcSD9RhcL7p0+flu2vAPuVXeuOilBx5ztPpqBLH6EKkbuaB5fT+WluMp2YHF2CbwnHvyx5FweBthyagHCtX794+eLrz56/LmvIapSdNTEWRZ5nShGcvppsSRHqgQyj84i4T0DHZgxVYmFHY6ceYead3ewu5Heby0Du4t3yMs4X7HGwnIzmB7scPwT5Zr9G6ngmMdeCMwwiCs0AdFl3bstoC5wBS1+ECsLL8OcIhVWLTBhkl6v8FBZDDBeADTpKV3MVIqf0NVq2gIbRdpEzPfjYynNG03HdELfTJuIGFxSbtotyolkb0W0pDozD1QwYNgNW/NDvtKwTz8oYvEsRGoFMdAzzeL1USbattUKuWqsi+12crrPACLpg+ivR7kTGjl32TCz5gNAY0Q8HFdMOlbqvwqowjsIjxkVsZbJb8bBJkDSbUJP00DtxUPAU2MXOOFzmNqX3plj0cQRFQRkOcHKGDFZ6JgVNAjNlNmyiX1bc8U3MZowss15npzTm6bdbELVskgkeyE3O2v3uYwFP/zHiIgJGCc0b+83OkQNeYhAqo1tM5JmX1zUFDU+gq3tidabZ53BdgOCGMtdkAnTMAQc8CY3qtoigspdyCsvliZmSW9CPTG6GDhoQkxjzA96fiLNAFVO0CCRwl8yBRpbov2LjzdYxGgrs4e1GqsKDDCLhpyxNZNQphwIWJGGiDuioVSxg8qgGE9DATg0okcUPWfWDAj+Oqe14KnbxKBVfbIImAQJc6zwpJ4ssIlBXSwkK4XeyXfJQnmmb4W4TdK8ypKmW0+5YoZoDjzo4amCgzUgJAHBHYBbK8Gl27lik90/deobtmReROK/PJRDdlZkKoybi+S1IF7PkyhbuHOlQesyqSo9B56jzhGjeefrxY/52j48+hp3s08edYWhh/VHf2weuo3sCq9l5WoHUPF4B29MRIBbEZ3iRsBkNa2isgvkaBAPJpOQ5tJB8WaVNz7zX6R5qhc4yDeIn+xrqk3KTUxuz3pgkPFH/YlbQJNrXa9ciB+raJZUrVUiWM3HIHbxp4XAwI1ZzIPAaXw4dM/CsmB3Ai2Jmbq7UIyNL0Ks5PvHkwN9BM+vBuoytlF6BsVV143pprGNQ1GEENhGBGrhEqSmUTv7EGElmdggj1VHBpdfbZkddTDAHufYagGpmAXatX3T2cpD1YjEu5Hh7kjtURM5+oDVCq0BhjCuUVbxvlQdaRZHU7KvJgnbnG9K9ydtg8BLirFT8dAY9AlqxSICM1GancP0/T8DhMBRvi+HvCuWQfz6GyTBMYM4dywfqg1aG98iCLsYsSRmC2AinVCbBicRgh7uqCM0EO/JYMqEZzciqHXU/Pv44bCFyFnU7mABGiyidrXw9voQzT/BzcCQpWPOtsgtGhBDNGKg0NiF6DN7qqV+GKG6l21UAZe2MTqji4tM8k4g0tImWV0gVYQc9gpF1HC0pRfLPem3NOV2fjxjiIoGG806Jt1buCwAbylX2NG2UYXAjhrPrp2s1Drc5MFnPNDG+u/XmesbxMCYzI+pxtlh/hKcA6YpHmDFYSxPkS6IAiQ0dkGi6NhaKZHFYWwIgZeJiOfDthGmU4qrNprj7ch68/+JZZ2MVpOu82K4SKmtKMjI19nt2piN69+yPacoAFxYOEucim9bUMNKDhH6SUKEZhlBUsERdC/EvaRpZFh0BIYp+r65DfbfAhSqLJdvFXYV0Hpkh6EPnSZsE4alDpgwgQGHP3TOCyeYQUNW1rEQF+6xeVIf+IUhYQTVYuZDhyfvoR2d+j9B7Fk0m2Vl/029/MKYU0UCsn9toumWg5599e0V7Z2y2EMJ9xBf32g4/dZ7ugjE4smyLn8EnBcoG44+icdUSMfZQGyOcLVYLqUVgxZdO1MuIFhwJFVcUQyz6FPA7DoTgjJeMSLHTIj356NYB13jKrCXou3H3WxSeMW64/taBcXzRjmHbyO0qWpDi5FDvXInbDoliaDxEDhHyTEIGbRdb75nmyThJXrYZ4jtpUChkaDjuT6xWRboywS9o8a7R9ndj1jN4u5c8pKaPoYtyiX+9RiR37+2X7+izvKC3tBe9wVoZ10cJMLU6RDh5sfYUI2XeEyT0BW2diolKoC7oi7DaW4mLz9+WhXDVk0ar3MObRldBQjyZJUqtA810DRV+LKbAsrR0f23Zpo1HZQZe8y2Ua2kWmOJgYqhjHq3fOlEm1UVz1yPGXTBZrLBub629OUydMhH9CgkzTGvNsrHwJlFyxkQhomHNNFFbpMcHxWsQf6oXOxwzlbsiP6dNcICGU66wiazYlF4pQao5heyAfuZ/ZRDqSiwor73gyurnEcY+FvsWM6Vr2u4qSmI9YPBrgpVp1BV1gwKqu9EINyqyty2Kq3KxNnB8Tnrto+l1a7l55FSaLMVmD4RwuYnnQuAQPX0BtnpRJYeXSvxI8tAWptxrHc2uJTHXtAIYZU7i24tFKfLlBumXRLq+hNZ5tW7Gw4E+4EibMw0meTZL7tH7zI9W28UCFlr9tH918vq6N1n0GSg1Xrumi0XjZzOplGF3x/2r7657V19e92ZcNkS3TebUu44iBo2GyIgnHOj+JKX0sTGBe4ut0ZghsOIVvvBw3RAWNe13GuZkvNyePeq2N3tOCr8ZaeewPyBXywFgwI3I1DS872gRXWL/Z3YW+xpaRhtxCu9ZjrBIiJAgFxyGIBJH9Y6k7Wtl0pMjEeaFF13E9GY7VCzGEjAANPCSIWcleZLKonSOb2lHTs4omkjhsb6CbL6V94gIRu9ocOvmgKUnPVsmSjuYUz29sWiCXZO3tbC2huhYsThfK/ilB1V9szINInS5wvMyry4QhgxJqhHEyaFz2z7zJZvfJ/GdK47cbiSHlxtMjBEAp/gOSt/f5rPmE73tzKpuWmo7qHOp28bJVQbu/zSyP2R8gk0ZWe5Q4sp9//0hYsuRdB9+vzJj2cP+3GvXK9uC4GORLncWzcqgA7LNdG77I7s0J1p9o40t12ezhfK7/gNikIbKtXzEcHdN9fstrXQJChiXoIYmCTLekQw2c4rvWWGzunW1hPZohbn4lgyCkSIkahH2b2klBdapGYI/iCB5XwaPLbnT5AzBMgIqJY+O2/yHzOVu5rHJynazpMkKF+5fnfba3SlIydL8ZvLbuIIOiETcfpwilnjmHLmxPa1IcUrUKy/W98A9WnN0Kpg4L61Yrd3scG+Iv6oh7Nw6ljd2M6zNN32Sd8MAFkE7NO6N+wI9ULTHP67uAeVuE8Dz7fX3qytnhSME/GjiE0Ru2jhkoUPv0AY3Ojw/HLoa1ENqUBkr5PsVt9QzxlciqOwHHVY0hrTx0GWijtt2ySebT8j0ldvAhpyDqGa4LBnGf5ePNgpWANiYDQZGrlExIqotUMWkSFUxtOPfZ7l94AQcsJ502poaPw8b5lXtpKsOmRXr5oPyAy7qp9TXl3mqiovCaUI80N3vtYgLQs1asVrCDccxL0xBBb9rBTG1FXl0YyOKlZIKi3mLh3SwmBeWuA5PWB30vFDYHVHn4Kj6jkRCjgJGCfeQzyJ4CMNCrfcSYQD/3TcnTuvU3nyMj1tR9WzKa1XTDN3NjvVIBn9QHZWFnjMqa9FhBoXmP2xM0u5UcIrNyPKpU0dkl9txaW56R61SCCFB0sSLk6PuVcdq2qRbgnmttBZJnGb1cdnj38KIvtyMdJbE0avqGi1Fs4N5Gd2qLY7C8yISR2mnFt5Ks3bt2cpIc2ZMvPNv09g42OgoGjZSL4JDcasi9EBDth5vT4tDTONUYtzC826yPmNw7z/RGQ9aR/nGEYrrMXbOT5FRX7RfDlioI7gg32ENWeFL/p5q3gk+HHbypTmSRIM3iZNFkHHtqQcQIX3XqFOd/NNqvn7OisJxRKUq+VTleE/IwJ6Q5+UZ3R8nky3+98s963y4qVh4NAZC6DfiH/okxDBkzaH9bB/3ECHICIL4/qQ3j5bLCHq9p08a0P2v03HEOApxH+c3dUS9cdZvykO67IuX/XgpCV0dABdHv5DhcDKvhALcg5dJrMRFhD3kac5mBXcDA3HJu4fAClxwcjGaLYUXAintKmtMJFRBTd8d3YtXnxn6qmY2iJ1Go5MTWj8d2U9h5aIr0fh7FI/7yXJuXcJKO181ulGcFHf1Xs3dki2ZmCW4kOK3elCOtzNRRAsQrVAJlxdwNnusbcxGRyVGqNMS93FMcgfxwOUkTKNuPsROO6N02gAQhgWhZ3Q+DGZGu2+QdVr+64YAwiNd15AroZntSnxhw1k57SpY8+VmdFYVdAgayC3ZUh94y4q84K9BOoeBLlFsLwrdR1Bqbt/fXV/gkint2LsT9nBokRy42Lj7+naPvvbe757Ux17cZhWSugWS1f4dsBpX2WmjFeLQcUS/qQjoV44hw9hBdBGgr0DMUMu9062pxRf2pGaeLBvjjeOwak4u8wkrnEtv1MNVPKvAXa5zo3zfRcs3uwi9MRwQPeWr6kIO1Ph3wRBrbTy21jv+8su52Ny4W7sAkSipBD5188xU1Imu0wdZ8Ia6fpgTUw18GHkMUx8DPYSl3NmP0pYbuFSsLOe3G2urSc9+Zxx+25I2v9oqfVE6vISKJ014H7N76Ocm6q1QuLjxOi68DpzkGvXwD7vmbO8U8BKgYig6y1bbRG2hCpMdhGIC1j7ando571D8LA76fKfH77tdg7m2Y/R28p1xpZwwdlaXOpsGaTwTgNaXlbJfir3hY2aeOqW/lITwdkhjPQrWz4DY2o998hv8j7VfX/SDC1C9S/x/IY6r53zhA7jAOku8Y6Tw/jte6RDg8GDtToFhlhfbb0uptrvWtE9NGksTw9ttnN9jI6h4kktAYFhjPNo9w0q2vh6c93XL/dKkJKG1X/VLRpZJxf64ZTPsG1/ZqdrScn279FXW4dr2ddAwjbnDtvcem23SuWWH2enr/eHOTx7ePg2p3DB2rEWCuV0Yq9bgy2qmXkNM7t2HjVJGodwKDhca5uxtyaFibvPGoNDFZ7+isXf9a8vLtYaSgN6URyV24WMJT9iux0jRb/d4SpC9q9X1w6vV+BrDvNKIoSvOiSm/DG/5qFYhftrBN/YOe6Hdbo14isfaynhD1aY+WSC+Og0yi/TziKENnxXs3FvwZ4EbPqDyD3q0abLWls/cr34qT1/5OC5jp5gv35bDHK9TdTox/tpXuXPojJxDpwy3L7F/nMB/RY62VYu+uCM9MsfwrnRofTcmc36FRbQ1l4E5kIMZEw0lMJi/76Yobkw767jrEjWsN22gxLeKX/4c/L1EzbDm+APbi3EmgsWMudZLqJuFEImHcMwdDnrHw12p3GTxfkju7SzbE7jngyBLG64qWAWzqEZxwK4bGxSitPFqFK70INjGRwpWX1UuHAlA8TPfWpM5pmuO0RoSDMrbcGi6jur9PyxUkKRl0RjEdTfW09v3hkybahAxPfLelU73ReNlHOkBFCzDCGr6Zf/q3eAwOhyKNqrHl7F5aUBZlG+gv3n17Unz5Nvnr7/FQ8PfTyWhDJpF07ifQ8/Um8qduGFTsMCwyOlfmSgfUz2QYRQrXUrho0EE0efQOe4VDM6I3QK8HvrPbJRjTk0UCM8kALJJ1gZsedqjz4xqoL/qXxFpFX7XvbM+oAHF1y3jVSTgmLuPnTF3H1dgAQDAcaEPfILAhNbBV7L0CsnFejulhW3/BS1fe9++6je7vZevn/c76LWqRKu1eadKDSvEwLH97uBsODh7O3xWrTzocJHvbkG/Fk5T5seDzIJXPjqklKcmjeALcLkgyfYr4+XWMlVVSWnssBZr2YJ+s1PIN0xFh1CYJXo26Bao2e65QNyLjh+kgtxB3plv5n/1ulX9bvp1o4bPx7ehh94OoSBoldiB58GyXV0PsyZI5lLRlAuDNN9NwMnfz0/xA+zRzdNTpe1xL8ppqojABr0Udj6AQf/ocXuw3Ns2sCAbIi9aJhcY94YzP+qpaTc4csDTmjv1ZzCIyBtIXoJVSfvyi46fPB1EmF+BCBUuT4Vnzzh81VUnq7fPtFO3x+5f12Mh5oiqBFYYgtbyUeT/BNXgRwYvRcYhASDExWNBSyhv5iiezWBu0xAaHX6JlXsVw3fqkWWK+5SHf6XOM7EyDfclauyoM0tt9S1z+3B9Z8lCmQGYrnna9W6PmALlCc2zzuDRsBMb5ZsTMUG5JTLKPnUodg33rYQi5h/T87w4Ywte6aDqlDzzP9VI7MZJveddSYVDWV0egp3ZNeTRwZWVxgwORYIiUfhtfJ6dXXhVfr5FoqTNSScvBofCiR1WmDQ0s6taCP3yzNcGdkMdq/ek9edWl3Vw6xeWERTtjPPRk91YZBeW1XJ8v2HHxnMVX4zVUdkRDX2dwj59hP2IQDUAvXNue94VJo+LOAB63OrOrjOkFLpLCVd5fYWe7fck7Ae+oLg057hG+sdqM267xNYZjcTCfIQYpbCQGxnu33wIBdF0wRGYL09EFHvZlNbzdL6lCf4rvqXGQDGCU8J0OopMXuCr7JMhSOs5inf+LfWaTYEoo0stNn3/X5/gExYSOhtrZr64Ddcu2tityg9yh3e0pifR3pE0SRIooDVuYfzy923tSPyOatG9LTKux74WCUE4mhBmLJ8ZuPFT6FZnHLhazX4k4gqr09X4IUg0cUQqUUSQzIfQjRRe2FEHIruOWoIkoX52T8qoO335mfSo5SimJIRv/64o5cYmz9ZiDBvng7g79MQOTrTmplKhN2/wU4A1fTlGqU/lDbli+2CVx8K8UTBqmwj5+Ty5VKTWKsd8bfibf+OXPYUH/xdzw3gTKIoAAA==")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».